# AET: initial price and shape analysis

**Question:** what within-hour patterns can we learn from the supplied prices alone?

This notebook uses the reproducible analysis script. Complete-file checks verify structure; all price exploration stops on **31 August 2026**. September remains excluded from plots and model selection. No external sources or predictive models are used.

Install `requirements-analysis.txt` in your notebook kernel environment before running. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown, Image

root = Path.cwd().resolve()
if not (root / 'analysis' / 'initial_analysis.py').exists():
    root = root.parent
assert (root / 'analysis' / 'initial_analysis.py').exists(), 'Open from the repo root or notebooks directory'
sys.path.insert(0, str(root))
from analysis.initial_analysis import run_analysis, load_and_check, prepare, centre_predictions
import numpy as np
import pandas as pd

results = run_analysis(root / 'Data/train_prices_15min.csv', root / 'analysis/results')
tables = results['output_dir'] / 'tables'
figures = results['output_dir'] / 'figures'

## 1. Quality and coverage
The loader checks the exact expected date/zone/hour/quarter grid and raises an error on missing keys, unexpected keys, duplicates, missing values or non-finite prices. DST transition dates are excluded by design. Negative prices remain valid observations.

In [ ]:
display(pd.Series(results['checks'], name='Check result'))
display(results['summary'].round(3))

## 2. Prices and deviations
The target is the quarter-hour price minus its hourly average. Inspect the full price tails and descriptive deviation QQ plots. Pooled deviations are not the same as conditional model residuals; these plots alone do not select a probabilistic likelihood.

In [ ]:
display(Image(filename=str(figures / '01_price_distributions.png')))
display(Image(filename=str(figures / '02_deviation_qq.png')))

## 3. Recurring shapes and sawtooth patterns
Look for differences by zone, delivery hour and quarter position. Increases within an hour can reset at the next hour boundary, so smooth interpolation is not necessarily appropriate. Example dates are selected by median daily flat-baseline error, not visual impact.

In [ ]:
display(Image(filename=str(figures / '03_shape_heatmaps.png')))
display(Image(filename=str(figures / '04_representative_days.png')))
display(Image(filename=str(figures / '05_flat_baseline_patterns.png')))

## 4. Candidate features: curve slopes and calendar lags
All target-day hourly prices arrive together; neighbouring hourly prices are permitted. Lag joins use calendar dates rather than row offsets. Correlations and copied-lag errors are exploratory diagnostics, not fitted-model validation results.

In [ ]:
display(Image(filename=str(figures / '06_slope_and_shape.png')))
display(results['slopes'].round(3))
display(results['lags'].round(3))

## 5. Illustrative valuation
A 1 MWh early-delivery profile puts all energy in Q1; a late-delivery profile puts it in Q4. Its absolute valuation error in EUR equals the absolute price error for that quarter multiplied by 1 MWh. These are transparent illustrations, not actual AET positions or trading profits. Equal volume in every quarter is valued exactly by the known hourly average.

In [ ]:
display(pd.read_csv(tables / 'illustrative_flat_valuation.csv').round(3))
display(Markdown(f'Exploratory equal-zone flat MAE: **{results["summary"].flat_mae.mean():.4f} EUR/MWh**.'))

## 6. Constraint and next steps
The centring layer must preserve the hourly average for every prediction. The small check below covers negative prices and non-rounded hourly means. An eventual submission also needs key validation and a CSV round-trip check.

Next: templates and compact CatBoost, using October–June for initial training, July–August for development and September for one final assessment. No BNN or calibrated confidence layer is needed for this initial analysis.

In [ ]:
hourly = np.array([100.0025, -20.0075, 0.0])
raw = np.array([[-15., -5., 5., 15.], [3., 2., 1., 0.], [2., -1., 7., 4.]])
predicted = centre_predictions(hourly, raw)
np.testing.assert_allclose(predicted.mean(axis=1), hourly, rtol=0, atol=1e-10)
display(pd.DataFrame(predicted, columns=['Q1', 'Q2', 'Q3', 'Q4']).assign(hourly_mean=predicted.mean(axis=1)))
print('Constraint check passed.')